# Qwen 8B + 14B + SVC OOF stacking diagnostic

SVC uses 5-fold GroupKFold by prompt_id. Qwen scores are extracted once from the published checkpoints; because those checkpoints include stage-2 training on all host rows, the report explicitly labels the Qwen component as frozen-checkpoint rather than pretending it is cross-fitted. We only promote a stack if the measured grouped diagnostic is positive.

In [ ]:
import gc, glob, json, os, re, time
import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).
W_SVC = 1.021
LORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for
    "lora8b_v1":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),
    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
}
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # retained for diagnostics; contribution disabled below

# Column order of the checkpoints' heads (checked against each checkpoint's meta.json below).
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))   # every member adds w * log p here
print("train", train.shape, "test", test.shape, flush=True)

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC


In [ ]:
# SVC GroupKFold OOF plus full-test scores.
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 180000, 1.0), ("word", 1, 3, 120000, 0.7), ("shape", 2, 6, 45000, 0.5), ("function", 1, 3, 35000, 0.5)]
def shape_text(t): return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)
def function_text(t): return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT" for w in re.findall(r"\w+|[^\w\s]", t))
def make_features(fit_texts, fit_y, val_texts, max_features_scale=1.0):
    A=[]; B=[]
    for view,lo,hi,mf,weight in VIEWS:
        prep=shape_text if view=="shape" else function_text if view=="function" else (lambda s:s)
        analyzer="char" if view=="shape" else "word" if view=="function" else view
        opts=dict(analyzer=analyzer,ngram_range=(lo,hi),lowercase=False,sublinear_tf=True,min_df=2,max_features=int(mf*max_features_scale),dtype=np.float32)
        if analyzer=="word": opts["token_pattern"]=r"(?u)\b\w+\b|[^\w\s]"
        v=TfidfVectorizer(**opts)
        A.append(v.fit_transform([prep(t) for t in fit_texts])*weight)
        B.append(v.transform([prep(t) for t in val_texts])*weight)
    return hstack(A,format="csr"),hstack(B,format="csr")
def score_to_prob(z):
    z=z-z.max(1,keepdims=True); e=np.exp(np.clip(z,-30,30)); return e/e.sum(1,keepdims=True)


In [ ]:
# Grouped OOF for SVC; public pool is fixed auxiliary data in every fold.
pool=pd.read_parquet(find("pool_v1.parquet")); pool_text=pool.text.astype(str).tolist(); pool_y=pool.model.map({l:i for i,l in enumerate(LABELS)}).values
svc_oof=np.zeros((len(train),K),np.float32); gkf=GroupKFold(5)
for fold,(itr,iva) in enumerate(gkf.split(train,y,train.prompt_id),1):
    Xtr,Xva=make_features(train.text.iloc[itr].astype(str).tolist()+pool_text, np.r_[y[itr],pool_y], train.text.iloc[iva].astype(str).tolist(), 0.55)
    m=LinearSVC(C=0.5,class_weight="balanced",max_iter=4000,random_state=fold).fit(Xtr,np.r_[y[itr],pool_y])
    svc_oof[iva]=score_to_prob(m.decision_function(Xva)); del Xtr,Xva,m; gc.collect(); print("svc fold",fold,flush=True)
# Full SVC fit for test.
Xtr,Xte=make_features(train.text.astype(str).tolist()+pool_text,np.r_[y,pool_y],test.text.astype(str).tolist(),0.55)
m=LinearSVC(C=0.5,class_weight="balanced",max_iter=5000,random_state=42).fit(Xtr,np.r_[y,pool_y])
svc_test=score_to_prob(m.decision_function(Xte)); del Xtr,Xte,m,pool; gc.collect()
print("svc oof BA",balanced_accuracy_score(y,svc_oof.argmax(1)),flush=True)


In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """First `keep` decoder layers, split over both T4s when it does not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    hd = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd
               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." in k:
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z (n, 8 * hidden) and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together = less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()
    del model
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}


In [ ]:
# Extract frozen-checkpoint Qwen scores for train and test in one pass per backbone.
all_text=train.text.astype(str).tolist()+test.text.astype(str).tolist()
q_probs=[]
for ckpt_name,mc in LORAS.items():
    ckpt=os.path.dirname(find(f"{ckpt_name}/meta.json")); Z,heads=lora_features(ckpt,find_base(mc["base"]),all_text,mc["bs"])
    q_probs.append(score_to_prob(Z@heads["plain.weight"].T+heads["plain.bias"]))
    del Z; gc.collect(); print(ckpt_name,"scores ready",flush=True)
q8_train,q8_test=q_probs[0][:len(train)],q_probs[0][len(train):]
q14_train,q14_test=q_probs[1][:len(train)],q_probs[1][len(train):]


In [ ]:
# Search conservative scalar weights. Qwen is frozen-checkpoint, SVC is true GroupKFold OOF.
# The grid is intentionally small to reduce meta-overfit.
weights=[0.25,0.5,0.75,1.0,1.25,1.5]
rows=[]; best=None
for ws in weights:
  for w8 in weights:
    for w14 in weights:
      z=ws*np.log(np.clip(svc_oof,1e-9,1))+w8*np.log(np.clip(q8_train,1e-9,1))+w14*np.log(np.clip(q14_train,1e-9,1))
      ba=balanced_accuracy_score(y,z.argmax(1)); rows.append((ba,ws,w8,w14))
      if best is None or ba>best[0]: best=(ba,ws,w8,w14)
rows=sorted(rows,reverse=True)
print("best grouped diagnostic",best,flush=True)
print("top 10",rows[:10],flush=True)
# Use the best stack only if it beats the original reference weights on the same diagnostic.
ref=(1.021,0.226,0.429)
ref_ba=balanced_accuracy_score(y,(ref[0]*np.log(np.clip(svc_oof,1e-9,1))+ref[1]*np.log(np.clip(q8_train,1e-9,1))+ref[2]*np.log(np.clip(q14_train,1e-9,1))).argmax(1))
print("reference diagnostic",ref_ba,flush=True)
chosen=best if best[0]>ref_ba+0.002 else (ref_ba,*ref)
print("chosen",chosen,flush=True)
_,ws,w8,w14=chosen
ztest=ws*np.log(np.clip(svc_test,1e-9,1))+w8*np.log(np.clip(q8_test,1e-9,1))+w14*np.log(np.clip(q14_test,1e-9,1))
out=pd.DataFrame({"id":test.id.astype(str),"model":np.array(LABELS)[ztest.argmax(1)]})
out.to_parquet("submission.parquet",index=False)
Path("stacking_results.json").write_text(json.dumps({"best":best,"reference_ba":float(ref_ba),"chosen":chosen,"note":"Qwen scores are frozen-checkpoint, not fold-retrained OOF"},indent=2))
